# Reachy Mini — Local AI Building Blocks (Explainer)

This notebook is a **concept companion** to the hands-on tasks in [`lab/LAB.md`](./LAB.md).
It shows the three local-AI pieces the lab is built from — the **LLM** (chat), the
**VLM** (vision), and **Piper TTS** (offline voice) — in isolation, so you can
poke at each one **without the robot, camera, or any motion hardware**.

> 🤖 This is *not* a replacement for the interactive robot tasks. The booth tasks
> stay as the `python lab/emo_v*.py` scripts. This is a "peek under the hood".

## What you need

- **Run this notebook from the repo root** (so the relative `models/` path resolves).
- A **local Ollama server** running at `http://localhost:11434`.
- The two models pulled:
  ```bash
  ollama pull qwen3.5:0.8b   # chat LLM (Tasks 1–2)
  ollama pull qwen2.5vl:3b   # vision VLM (Task 3)
  ```
- **Piper** installed (already in `requirements.txt`) and a voice model in `models/`.

> ℹ️ **Streaming note:** the lab scripts stream tokens from Ollama for snappiness at
> the booth. Here we use plain **non-streaming** requests (`"stream": False`) and just
> read the final JSON — simpler to read in a notebook. The payloads otherwise mirror
> the real scripts exactly.

## Config + Ollama health check

Model names and paths, plus a tiny helper that pings Ollama's `/api/tags` and tells
you whether the server is reachable and which of the two models are present.

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434"
CHAT_MODEL = "qwen3.5:0.8b"
VLM_MODEL = "qwen2.5vl:3b"
PIPER_MODEL = "models/en-us-blizzard_lessac-medium.onnx"


def check_ollama():
    """Ping Ollama and report which lab models are available. No traceback if it's down."""
    try:
        resp = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        resp.raise_for_status()
    except requests.exceptions.ConnectionError:
        print(f"❌ Ollama not reachable at {OLLAMA_URL}. Is `ollama serve` running?")
        return
    except Exception as e:
        print(f"⚠️ Could not query Ollama: {e}")
        return

    print(f"✅ Ollama is reachable at {OLLAMA_URL}")
    names = [m.get("name", "") for m in resp.json().get("models", [])]
    for wanted in (CHAT_MODEL, VLM_MODEL):
        present = wanted in names or f"{wanted}:latest" in names
        mark = "✅" if present else "⚠️"
        hint = "" if present else f"  (pull it: `ollama pull {wanted}`)"
        print(f"   {mark} {wanted}{hint}")


check_ollama()

## Building block 1 — Local LLM (chat)

This is the conversational brain from **Task 1 & 2**. We POST to `/api/chat` with a
`system` + `user` message pair, mirroring `emo_v2.py`'s `_get_ollama_response_async`
(same `think=False`, `keep_alive`, and `options`). Everything runs locally.

> 🔎 Changing the `system` prompt below is **exactly** the Task 1/2 `ROBOT_PERSONA`
> edit — it's what gives Reachy its personality.

In [ ]:
ROBOT_PERSONA = (
    "You are a cute desktop robot assistant. Respond with enthusiasm and warmth, "
    "in two or three short sentences. Keep it brief and conversational."
)


def chat(prompt, system=ROBOT_PERSONA, model=CHAT_MODEL):
    """Non-streaming local chat, mirroring emo_v2's /api/chat payload."""
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        "stream": False,
        "think": False,
        "keep_alive": "30m",
        "options": {"temperature": 0.8, "num_predict": 120},
    }
    try:
        resp = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=300)
        resp.raise_for_status()
    except requests.exceptions.ConnectionError:
        print(f"❌ Ollama not reachable at {OLLAMA_URL}. Is `ollama serve` running?")
        return None
    except Exception as e:
        print(f"⚠️ Chat request failed: {e}")
        return None

    data = resp.json()
    if data.get("error"):
        print(f"⚠️ Ollama error: {data['error']}")
        return None
    return data.get("message", {}).get("content", "").strip()


reply = chat("tell me a joke")
if reply:
    print(f"🤖 {reply}")

## Building block 2 — Local Vision model (VLM)

This is Reachy's eyes from **Task 3**. We read a JPEG, base64-encode it, and POST to
`/api/generate` with an `images=[...]` field and the vision model — mirroring
`emo_v3.py`'s `_describe_scene`. **No camera needed here**: point `IMAGE_PATH` at any
image on disk.

> 💡 There's no bundled sample image in this repo. Grab one from the robot with
> `python lab/emo_v3.py --save-frame look.jpg`, or just set `IMAGE_PATH` to any
> `.jpg`/`.png` you have.

> 🔎 Changing `VISION_PROMPT` below is **exactly** the Task 3 edit — it's what Reachy
> is asked about the scene.

In [ ]:
import base64
import os
from IPython.display import Image, display

# No sample image ships with the repo — set this to any .jpg/.png you have.
IMAGE_PATH = "look.jpg"

VISION_PROMPT = (
    "You are a curious desktop robot looking through your own camera. "
    "In one or two short, upbeat sentences, describe what you see right now."
)


def describe_image(path=IMAGE_PATH, prompt=VISION_PROMPT, model=VLM_MODEL):
    """Non-streaming local vision, mirroring emo_v3's /api/generate + images payload."""
    if not os.path.exists(path):
        print(f"⚠️ Image not found: {path}")
        print("   Set IMAGE_PATH to any .jpg/.png, e.g. one saved via")
        print("   `python lab/emo_v3.py --save-frame look.jpg`.")
        return None

    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("ascii")

    display(Image(filename=path))

    payload = {
        "model": model,
        "prompt": prompt,
        "images": [b64],
        "stream": False,
        "keep_alive": "30m",
        "options": {"temperature": 0.5, "num_predict": 120},
    }
    try:
        resp = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=300)
        resp.raise_for_status()
    except requests.exceptions.ConnectionError:
        print(f"❌ Ollama not reachable at {OLLAMA_URL}. Is `ollama serve` running?")
        return None
    except Exception as e:
        print(f"⚠️ Vision request failed: {e}")
        return None

    data = resp.json()
    if data.get("error"):
        print(f"⚠️ Ollama error: {data['error']}")
        return None
    return data.get("response", "").strip()


description = describe_image()
if description:
    print(f"🤖 Reachy sees: {description}")

## Building block 3 — Offline TTS with Piper

This is Reachy's **offline voice** from **Task 2**. We load a Piper `.onnx` voice the
same way the repo does — manually read the `.onnx.json` config, apply the legacy
`PhonemeType.ESPEAK` → `espeak` fix, build a `PiperConfig`, spin up an onnxruntime
**CPU** session, and wrap it in a `PiperVoice`.

In a notebook we can't rely on speakers, so instead of playing through `sounddevice`
we synthesize to a temp WAV and **embed it inline** with `IPython.display.Audio` so
you can play it from the browser.

> 🔊 This is the fully-offline voice used in **Task 2**. The **Task 1** contrast is
> `edge-tts`, Microsoft's *cloud* neural voices — nicer, but needs internet.

In [ ]:
import json
import tempfile
import wave
from IPython.display import Audio, display


def load_piper(model_path=PIPER_MODEL):
    """Load a Piper voice exactly like emo_v2's PiperTTSEngine. Returns (voice, SynthesisConfig)."""
    try:
        from piper import PiperVoice, PiperConfig
        try:
            from piper import SynthesisConfig
        except ImportError:
            SynthesisConfig = None
        import onnxruntime
    except ImportError:
        print("❌ piper-tts not installed. Install with: pip install piper-tts")
        return None, None

    if not os.path.exists(model_path):
        print(f"❌ Piper model not found at: {model_path}")
        print("⚠️ Download one from https://github.com/rhasspy/piper/releases/tag/v0.0.2")
        return None, None

    try:
        with open(model_path + ".json", "r", encoding="utf-8") as f:
            config_dict = json.load(f)
        # FIX: replace legacy "PhonemeType.ESPEAK" string with "espeak"
        if config_dict.get("phoneme_type") == "PhonemeType.ESPEAK":
            print("🔧 Fixing legacy phoneme_type in config...")
            config_dict["phoneme_type"] = "espeak"

        config = PiperConfig.from_dict(config_dict)
        session = onnxruntime.InferenceSession(
            str(model_path),
            sess_options=onnxruntime.SessionOptions(),
            providers=["CPUExecutionProvider"],
        )
        voice = PiperVoice(session=session, config=config)
        print("✅ Piper TTS initialized")
        return voice, SynthesisConfig
    except Exception as e:
        print(f"❌ Failed to load Piper model: {e}")
        return None, None


_voice, _SynthesisConfig = load_piper()


def say(text, speaker_id=0):
    """Synthesize text to a temp WAV and embed it inline for browser playback."""
    if not _voice:
        print("⚠️ Piper voice not loaded — run the load cell above first.")
        return None
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp_path = tmp.name
    syn_config = None
    if _SynthesisConfig is not None:
        syn_config = _SynthesisConfig(speaker_id=speaker_id)
    with wave.open(tmp_path, "wb") as wav_file:
        _voice.synthesize_wav(text, wav_file, syn_config=syn_config)
    display(Audio(tmp_path))
    return tmp_path


say("Hello! I'm Reachy Mini, and this is my offline voice.")

## Where this fits

This notebook is a great **explainer / onboarding** artifact: it isolates the three
local-AI building blocks so you can understand and tweak each one on its own. The
actual booth tasks, though, stay as the `python lab/emo_v*.py` scripts — they own the
terminal, use `Ctrl+C` for clean shutdown, drive real motion/camera hardware, and
share an import chain (`emo_v3` → `emo_v2` → `emo_v1`) that a notebook doesn't fit
cleanly. Think of this as optional **"peek under the hood"** material that could live
alongside the lab, not a replacement for the hands-on tasks in [`lab/LAB.md`](./LAB.md).